# LeetCode 215 — Kth Largest Element in an Array bằng Python

Link đề: https://leetcode.com/problems/kth-largest-element-in-an-array/ · Độ khó: **Medium** · Chủ đề: Array, Sorting, Heap (Priority Queue), Quickselect, Counting Sort

Skeleton LeetCode:

```python
class Solution:
    def findKthLargest(self, nums: List[int], k: int) -> int:
        ...
```

Ghi chú: đề, ví dụ và constraints lấy từ bản mô tả gốc; mọi đáp án trong notebook đều được kiểm chứng bằng code và đối chiếu với một lời giải brute force `O(n^2)` viết độc lập (không dùng sắp xếp).

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: tìm **phần tử lớn thứ `k`** trong mảng **khi đã sắp xếp** — **tính cả phần tử trùng** (không phải phần tử phân biệt thứ `k`).
2. Biết **5 cách** từ chậm đến nhanh: **sắp xếp** `O(n log n)` → **min-heap kích thước `k`** `O(n log k)` → **heapify cả mảng + `k-1` lần pop** `O(n + k log n)` → **Quickselect** trung bình `O(n)` → **đếm (counting) / bucket** `O(n + R)`.
3. Hiểu cấu trúc **heap** (`heapq` của Python là **min-heap**), vì sao min-heap cỡ `k` giữ đúng "`k` phần tử lớn nhất", và cách đọc `heap[0]`.
4. Hiểu thuật toán **Quickselect** (một nửa của Quick Sort: chỉ đi vào **một** phía), vai trò của **pivot ngẫu nhiên** và **phân hoạch 3 phần** khi có nhiều giá trị trùng.
5. Đọc kỹ **constraints** (`n <= 10^5`, giá trị trong `[-10^4, 10^4]`) để chọn cách hợp lý, và tránh các lỗi kinh điển: dùng `set` (bỏ trùng), nhầm chỉ số `k` vs `k-1`, quên heap Python là min-heap, quickselect suy biến `O(n^2)` khi trùng lặp nhiều.

## 2. Đề bài & ví dụ

Cho mảng số nguyên `nums` và số nguyên `k`. Trả về **phần tử lớn thứ `k`** trong mảng.

Lưu ý: đó là phần tử lớn thứ `k` **theo thứ tự sắp xếp**, **không** phải phần tử **phân biệt** (distinct) lớn thứ `k`.

Bạn có thể giải mà **không sắp xếp** được không?

**Ví dụ 1**

- Input: `nums = [3,2,1,5,6,4], k = 2`
- Output: `5`
- Giải thích: sắp giảm dần được `[6,5,4,3,2,1]`; phần tử thứ 2 là `5`.

**Ví dụ 2**

- Input: `nums = [3,2,3,1,2,4,5,5,6], k = 4`
- Output: `4`
- Giải thích: sắp giảm dần được `[6,5,5,4,3,3,2,2,1]`; phần tử thứ 4 là `4` (hai số `5` được tính là **hai** phần tử).

**Ràng buộc:** `1 <= k <= nums.length <= 10^5`; `-10^4 <= nums[i] <= 10^4`.

In [1]:
from typing import List
import heapq
import random
import time

examples = [
    ([3, 2, 1, 5, 6, 4], 2, 5),
    ([3, 2, 3, 1, 2, 4, 5, 5, 6], 4, 4),
]
for nums, k, expected in examples:
    print(f"nums={nums}, k={k} -> kỳ vọng {expected}")

nums=[3, 2, 1, 5, 6, 4], k=2 -> kỳ vọng 5
nums=[3, 2, 3, 1, 2, 4, 5, 5, 6], k=4 -> kỳ vọng 4


## 3. Kiến thức nền

### 3.1. Sắp xếp và chỉ số: `sorted`, `reverse=True`

`sorted(nums)` trả danh sách **mới** tăng dần; `sorted(nums, reverse=True)` giảm dần. Phần tử **lớn thứ `k`** (đếm từ 1) nằm ở chỉ số **`k - 1`** của danh sách giảm dần, hoặc chỉ số **`-k`** (hoặc `n - k`) của danh sách tăng dần.

In [2]:
nums = [3, 2, 1, 5, 6, 4]
desc = sorted(nums, reverse=True)
asc = sorted(nums)
print(desc, asc)
k = 2
print(desc[k - 1], asc[-k], asc[len(nums) - k])   # 5 5 5

[6, 5, 4, 3, 2, 1] [1, 2, 3, 4, 5, 6]
5 5 5


### 3.2. Phần tử trùng vẫn tính từng cái

Với `[6,5,5,4,3,3,2,2,1]`: hạng 1 là `6`, hạng 2 là `5`, hạng 3 **cũng là** `5` (cái `5` thứ hai), hạng 4 là `4`. Nếu bỏ trùng bằng `set` thì dãy chỉ còn `[6,5,4,3,2,1]` và hạng 3 trở thành `4` — sai với đề.

In [3]:
nums = [3, 2, 3, 1, 2, 4, 5, 5, 6]
desc = sorted(nums, reverse=True)
print(desc)
print("thứ 3 (đúng):", desc[2], "| nếu bỏ trùng bằng set:", sorted(set(nums), reverse=True)[2])

[6, 5, 5, 4, 3, 3, 2, 2, 1]
thứ 3 (đúng): 5 | nếu bỏ trùng bằng set: 4


### 3.3. Heap và module `heapq`

**Min-heap** là cấu trúc cho phép: **thêm** phần tử và **lấy ra phần tử nhỏ nhất** rất nhanh (`O(log m)` với `m` là số phần tử đang chứa), và xem phần tử nhỏ nhất trong `O(1)`. Python dùng **danh sách thường** làm heap với các hàm của `heapq`:

- `heapq.heappush(h, x)`: thêm `x`.
- `heapq.heappop(h)`: lấy và trả phần tử **nhỏ nhất**.
- `h[0]`: xem phần tử nhỏ nhất (không lấy ra).
- `heapq.heapify(lst)`: biến danh sách thành heap **tại chỗ** trong `O(n)`.
- `heapq.heappushpop(h, x)`: thêm `x` rồi lấy ra phần tử nhỏ nhất (nhanh hơn push rồi pop).

Lưu ý: `heapq` là **min-heap**; muốn max-heap thì lưu **số đối** `-x`.

In [4]:
h = []
for x in [5, 1, 4, 2, 3]:
    heapq.heappush(h, x)
    print(f"push {x} -> heap={h}, nhỏ nhất h[0]={h[0]}")
print("pop:", heapq.heappop(h), "->", h)
print("pop:", heapq.heappop(h), "->", h)

push 5 -> heap=[5], nhỏ nhất h[0]=5
push 1 -> heap=[1, 5], nhỏ nhất h[0]=1
push 4 -> heap=[1, 5, 4], nhỏ nhất h[0]=1
push 2 -> heap=[1, 2, 4, 5], nhỏ nhất h[0]=1
push 3 -> heap=[1, 2, 4, 5, 3], nhỏ nhất h[0]=1
pop: 1 -> [2, 3, 4, 5]
pop: 2 -> [3, 5, 4]


### 3.3b. Vì sao min-heap cỡ `k` giữ đúng "`k` phần tử lớn nhất"?

Giữ trong heap **`k` số lớn nhất đã thấy**. Khi một số mới `x` đến:

- nếu heap chưa đủ `k` phần tử: thêm vào;
- nếu đã đủ `k`: so `x` với **phần tử nhỏ nhất** trong heap `h[0]` — nếu `x > h[0]` thì `h[0]` không còn nằm trong top `k` nữa, ta thay nó bằng `x`; nếu không thì bỏ `x`.

Cuối cùng `h[0]` là **nhỏ nhất trong `k` số lớn nhất**, tức đúng **phần tử lớn thứ `k`**.

In [5]:
h = [4, 5, 6]            # giả sử đã là heap hợp lệ cho k = 3
heapq.heapify(h)
print(h, "-> nhỏ nhất h[0] =", h[0], "(đang là lớn thứ 3)")
print("heappushpop(7):", heapq.heappushpop(h, 7), "-> heap", h)    # 4 bị đẩy ra, 7 vào
print("heappushpop(1):", heapq.heappushpop(h, 1), "-> heap", h)    # 1 nhỏ hơn h[0] -> chính nó bị đẩy ra

[4, 5, 6] -> nhỏ nhất h[0] = 4 (đang là lớn thứ 3)
heappushpop(7): 4 -> heap [5, 7, 6]
heappushpop(1): 1 -> heap [5, 7, 6]


### 3.4. Danh sách sinh (list comprehension) để lọc

`[x for x in arr if x > p]` tạo danh sách các phần tử **lớn hơn** `p`. Ta sẽ dùng nó để **chia mảng thành ba phần**: lớn hơn, bằng, nhỏ hơn một giá trị gọi là **pivot** (điểm tựa).

In [6]:
arr = [3, 2, 1, 5, 6, 4]
p = 4
greater = [x for x in arr if x > p]
equal = [x for x in arr if x == p]
smaller = [x for x in arr if x < p]
print(greater, equal, smaller)      # [5, 6] [4] [3, 2, 1]

[5, 6] [4] [3, 2, 1]


### 3.5. Mảng đếm (counting array) và độ lệch (offset)

Khi giá trị nằm trong một khoảng **nhỏ** (ở đây `[-10^4, 10^4]`, tức chỉ `20001` giá trị), ta có thể đếm số lần xuất hiện của mỗi giá trị trong một mảng `cnt`. Vì chỉ số mảng không âm, ta **dịch** (offset) giá trị: giá trị `v` ứng với chỉ số `v - min_value`.

In [7]:
nums = [3, -2, 3, 1, -2, 4]
lo, hi = min(nums), max(nums)
cnt = [0] * (hi - lo + 1)
for v in nums:
    cnt[v - lo] += 1
print("lo =", lo, "cnt =", cnt)      # chỉ số 0 ứng với -2
for idx, c in enumerate(cnt):
    print(f"giá trị {idx + lo}: {c} lần")

lo = -2 cnt = [2, 0, 0, 1, 0, 2, 1]
giá trị -2: 2 lần
giá trị -1: 0 lần
giá trị 0: 0 lần
giá trị 1: 1 lần
giá trị 2: 0 lần
giá trị 3: 2 lần
giá trị 4: 1 lần


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Với `nums = [3,2,3,1,2,4,5,5,6]`, `k = 4`:

1. Xếp **giảm dần**: `6, 5, 5, 4, 3, 3, 2, 2, 1`.
2. Đếm tới vị trí thứ 4: `6` (1), `5` (2), `5` (3), `4` (4) → đáp án **4**.

Đó là cách "mắt thường" của mọi người. Câu hỏi của bài là: có cách rẻ hơn sắp xếp toàn bộ không? Ta chỉ cần biết **vị trí thứ `k`**, không cần biết thứ tự của các phần tử còn lại. Mọi cách sau đây đều xuất phát từ việc **bỏ bớt công sắp xếp thừa**:

- **Heap cỡ `k`**: chỉ giữ `k` số lớn nhất, không sắp xếp phần còn lại.
- **Quickselect**: chia mảng theo pivot, biết đáp án nằm ở **một** nửa, loại nửa kia.
- **Đếm**: vì giá trị nhỏ (`20001` giá trị khác nhau), không cần so sánh.

## 5. Cách 1 — Brute Force / baseline: sắp xếp giảm dần rồi lấy `k - 1`

**Ý tưởng.** Sắp xếp toàn bộ mảng giảm dần, trả `nums[k - 1]`. Đơn giản, đúng, và thường là đủ nhanh với `n <= 10^5`. Đề hỏi "có giải mà không sắp xếp được không?" để khuyến khích nghĩ tiếp.

In [8]:
def kth_sort(nums, k):
    return sorted(nums, reverse=True)[k - 1]


for nums, k, expected in examples:
    print(nums, k, "->", kth_sort(nums, k), "(kỳ vọng", expected, ")")

[3, 2, 1, 5, 6, 4] 2 -> 5 (kỳ vọng 5 )
[3, 2, 3, 1, 2, 4, 5, 5, 6] 4 -> 4 (kỳ vọng 4 )


In [9]:
# Quan sát từng bước ví dụ 2
nums, k = [3, 2, 3, 1, 2, 4, 5, 5, 6], 4
desc = sorted(nums, reverse=True)
for rank, v in enumerate(desc, start=1):
    print(f"hạng {rank}: {v}", "<- đáp án" if rank == k else "")

hạng 1: 6 
hạng 2: 5 
hạng 3: 5 
hạng 4: 4 <- đáp án
hạng 5: 3 
hạng 6: 3 
hạng 7: 2 
hạng 8: 2 
hạng 9: 1 


**Độ phức tạp (Cách 1).** Gọi `n = len(nums)`. Sắp xếp (Timsort) **Time `O(n log n)`**; **Space `O(n)`** vì `sorted` tạo danh sách mới (dùng `nums.sort()` thì chỉ `O(1)`–`O(n)` phụ trợ nhưng làm thay đổi mảng đầu vào).

**Nút thắt.** Ta sắp xếp **mọi** phần tử trong khi chỉ cần biết phần tử ở vị trí `k`. Khi `k` nhỏ (như `k = 2`), phải tốn `n log n` để tìm top 2 là lãng phí.

## 6. Ý tưởng thuật toán / bước đột phá

Có ba hướng "bỏ bớt công việc":

**Hướng 1 — Chỉ giữ top `k` (heap).** Dùng min-heap kích thước `k`. Mỗi phần tử mới tốn `O(log k)` (không phải `O(log n)`), tổng `O(n log k)`. Khi `k` nhỏ rất nhanh.

**Hướng 2 — Chia để loại (Quickselect).** Chọn một pivot `p`, chia mảng thành `greater` (`> p`), `equal` (`== p`), `smaller` (`< p`). Đáp án nằm ở **một** trong ba phần, xác định bằng cách so `k` với `len(greater)` và `len(greater) + len(equal)`. Ta **chỉ đi tiếp vào một phần**, nên tổng công việc là `n + n/2 + n/4 + ... ≈ 2n` khi pivot "vừa phải": `O(n)` trung bình.

**Hướng 3 — Đếm.** Giá trị chỉ có `R = 20001` khả năng. Đếm số lần mỗi giá trị xuất hiện rồi đi từ **giá trị lớn xuống nhỏ**, trừ dần `k` cho đến khi `k <= 0`: giá trị hiện tại là đáp án. `O(n + R)`.

## 7. Các cách tiếp theo

### Cách 2 — Min-heap kích thước `k`

**Giải thích.** Duyệt từng `x`. Nếu heap có ít hơn `k` phần tử: `heappush`. Ngược lại, nếu `x > h[0]`: `heappushpop`/`heapreplace` để thay phần tử nhỏ nhất. Cuối cùng `h[0]` là đáp án (nhỏ nhất trong `k` số lớn nhất).

In [10]:
def kth_min_heap(nums, k):
    h = []
    for x in nums:
        if len(h) < k:
            heapq.heappush(h, x)
        elif x > h[0]:                    # x lớn hơn phần tử nhỏ nhất trong top-k
            heapq.heapreplace(h, x)       # pop nhỏ nhất rồi push x (một thao tác)
    return h[0]


for nums, k, expected in examples:
    print(nums, k, "->", kth_min_heap(nums, k), "(kỳ vọng", expected, ")")

[3, 2, 1, 5, 6, 4] 2 -> 5 (kỳ vọng 5 )
[3, 2, 3, 1, 2, 4, 5, 5, 6] 4 -> 4 (kỳ vọng 4 )


In [11]:
# Code nộp LeetCode cho Cách 2 (bản an toàn: worst case O(n log k))
class Solution:
    def findKthLargest(self, nums: List[int], k: int) -> int:
        h = []
        for x in nums:
            if len(h) < k:
                heapq.heappush(h, x)
            elif x > h[0]:
                heapq.heapreplace(h, x)
        return h[0]


print(Solution().findKthLargest([3, 2, 1, 5, 6, 4], 2))   # 5

5


**Độ phức tạp Cách 2:** **Time `O(n log k)`** (mỗi phần tử một thao tác heap `O(log k)`), **Space `O(k)`**. Nhanh hơn sắp xếp khi `k << n`; khi `k ≈ n` thì cùng bậc `O(n log n)`.

### Cách 3 — Heapify cả mảng rồi pop `k - 1` lần (max-heap bằng số đối)

**Giải thích.** Đưa `-x` của mọi `x` vào một danh sách rồi `heapify` (`O(n)`), khi đó phần tử nhỏ nhất của heap chính là **số lớn nhất** (âm của nó). Pop `k - 1` lần để loại `k - 1` số lớn nhất; phần tử còn ở đỉnh là lớn thứ `k`.

In [12]:
def kth_max_heap(nums, k):
    h = [-x for x in nums]           # đổi dấu để min-heap hoạt động như max-heap
    heapq.heapify(h)                 # O(n)
    for _ in range(k - 1):
        heapq.heappop(h)             # bỏ k-1 số lớn nhất
    return -h[0]                     # đổi dấu lại


for nums, k, expected in examples:
    print(nums, k, "->", kth_max_heap(nums, k), "(kỳ vọng", expected, ")")

[3, 2, 1, 5, 6, 4] 2 -> 5 (kỳ vọng 5 )
[3, 2, 3, 1, 2, 4, 5, 5, 6] 4 -> 4 (kỳ vọng 4 )


**Độ phức tạp Cách 3:** **Time `O(n + k log n)`** (`heapify` `O(n)` cộng `k - 1` lần pop, mỗi lần `O(log n)`), **Space `O(n)`**. Tốt khi `k` rất nhỏ (gần `O(n)`); khi `k ≈ n/2` cùng bậc `O(n log n)`.

### Cách 4 — Quickselect (chọn nhanh) với pivot ngẫu nhiên và chia 3 phần

**Giải thích.** Làm lần lượt:

1. Chọn pivot **ngẫu nhiên** `p` trong mảng hiện tại (chống dữ liệu "xấu" như mảng đã sắp xếp).
2. Chia thành `greater` (`> p`), `equal` (`== p`), `smaller` (`< p`).
3. `g = len(greater)`, `e = len(equal)`. Nếu `k <= g`: đáp án trong `greater`, **tiếp tục với `greater`**. Nếu `k <= g + e`: đáp án chính là `p`. Ngược lại: đáp án trong `smaller`, và là phần tử lớn thứ `k - g - e` của nó.

Dùng `equal` để xử lý **mọi giá trị trùng** trong một lần, tránh suy biến khi mảng có rất nhiều số bằng nhau.

In [13]:
def kth_quickselect(nums, k, rng=random):
    arr = list(nums)                           # không làm thay đổi mảng gốc
    while True:
        pivot = rng.choice(arr)
        greater = [x for x in arr if x > pivot]
        equal = [x for x in arr if x == pivot]
        g, e = len(greater), len(equal)
        if k <= g:
            arr = greater                      # đáp án nằm trong nhóm lớn hơn
        elif k <= g + e:
            return pivot                       # đáp án là chính pivot
        else:
            k -= g + e                         # bỏ g + e số lớn: tìm tiếp trong nhóm nhỏ hơn
            arr = [x for x in arr if x < pivot]


random.seed(215)
for nums, k, expected in examples:
    print(nums, k, "->", kth_quickselect(nums, k), "(kỳ vọng", expected, ")")

[3, 2, 1, 5, 6, 4] 2 -> 5 (kỳ vọng 5 )
[3, 2, 3, 1, 2, 4, 5, 5, 6] 4 -> 4 (kỳ vọng 4 )


Phiên bản **tại chỗ (in-place)** tiết kiệm bộ nhớ: dùng ba con trỏ `lt`, `i`, `gt` (kiểu **cờ Hà Lan**) để phân hoạch `nums[lo..hi]` thành ba vùng: `[lo, lt)` các số `> p`, `[lt, gt]` các số `== p`, `(gt, hi]` các số `< p`. Rồi so chỉ số đích `k - 1` với các vùng để thu hẹp `[lo, hi]`.

In [14]:
def kth_quickselect_inplace(nums, k):
    nums = list(nums)                          # làm việc trên bản sao
    lo, hi = 0, len(nums) - 1
    target = k - 1                             # chỉ số của phần tử lớn thứ k khi sắp giảm dần
    while lo <= hi:
        p = nums[random.randint(lo, hi)]
        lt, i, gt = lo, lo, hi
        while i <= gt:
            if nums[i] > p:
                nums[lt], nums[i] = nums[i], nums[lt]
                lt += 1
                i += 1
            elif nums[i] < p:
                nums[i], nums[gt] = nums[gt], nums[i]
                gt -= 1
            else:
                i += 1
        # bây giờ: nums[lo:lt] > p, nums[lt:gt+1] == p, nums[gt+1:hi+1] < p
        if target < lt:
            hi = lt - 1
        elif target > gt:
            lo = gt + 1
        else:
            return p


random.seed(2150)
for nums, k, expected in examples:
    print(nums, k, "->", kth_quickselect_inplace(nums, k), "(kỳ vọng", expected, ")")

[3, 2, 1, 5, 6, 4] 2 -> 5 (kỳ vọng 5 )
[3, 2, 3, 1, 2, 4, 5, 5, 6] 4 -> 4 (kỳ vọng 4 )


**Độ phức tạp Cách 4:** **Time trung bình `O(n)`** (mỗi vòng kích thước mảng giảm kỳ vọng một hằng số lần), **worst case `O(n^2)`** nếu liên tiếp chọn pivot xấu (xác suất cực nhỏ với pivot ngẫu nhiên). **Space `O(n)`** cho phiên bản dùng danh sách, **`O(1)`** phụ trợ cho phiên bản tại chỗ.

### Cách 5 — Đếm (counting) theo giá trị

**Giải thích.** Vì `nums[i] ∈ [-10^4, 10^4]`, dựng mảng đếm kích thước `max - min + 1`; duyệt giá trị từ **lớn xuống nhỏ**, mỗi bước trừ số lần xuất hiện khỏi `k`; khi `k <= 0` thì giá trị hiện tại là đáp án.

In [15]:
def kth_counting(nums, k):
    lo, hi = min(nums), max(nums)
    cnt = [0] * (hi - lo + 1)
    for v in nums:
        cnt[v - lo] += 1
    for value in range(hi, lo - 1, -1):          # từ lớn xuống nhỏ
        k -= cnt[value - lo]
        if k <= 0:
            return value


for nums, k, expected in examples:
    print(nums, k, "->", kth_counting(nums, k), "(kỳ vọng", expected, ")")

[3, 2, 1, 5, 6, 4] 2 -> 5 (kỳ vọng 5 )
[3, 2, 3, 1, 2, 4, 5, 5, 6] 4 -> 4 (kỳ vọng 4 )


**Độ phức tạp Cách 5:** **Time `O(n + R)`** với `R = max - min + 1 <= 20001`, **Space `O(R)`**. Đây thường là cách nhanh nhất **khi miền giá trị nhỏ** như đề này. Nếu miền giá trị lớn (ví dụ `10^9`) thì không dùng được.

## 8. TRACE từng bước

Ta trace hai thuật toán: **min-heap cỡ `k`** trên ví dụ 1 và **quickselect** (pivot chọn có kiểm soát để dễ theo dõi) trên ví dụ 2.

In [16]:
def trace_min_heap(nums, k):
    print(f"nums={nums}, k={k}")
    h = []
    for idx, x in enumerate(nums):
        if len(h) < k:
            heapq.heappush(h, x)
            action = f"heap chưa đủ {k} -> push {x}"
        elif x > h[0]:
            out = heapq.heapreplace(h, x)
            action = f"{x} > h[0] -> đẩy {out} ra, đưa {x} vào"
        else:
            action = f"{x} <= h[0]={h[0]} -> bỏ qua"
        print(f"i={idx}: x={x}: {action:<40} heap={sorted(h)} (h[0]={h[0]})")
    print("đáp án = h[0] =", h[0])
    return h[0]


trace_min_heap([3, 2, 1, 5, 6, 4], 2)

nums=[3, 2, 1, 5, 6, 4], k=2
i=0: x=3: heap chưa đủ 2 -> push 3                 heap=[3] (h[0]=3)
i=1: x=2: heap chưa đủ 2 -> push 2                 heap=[2, 3] (h[0]=2)
i=2: x=1: 1 <= h[0]=2 -> bỏ qua                    heap=[2, 3] (h[0]=2)
i=3: x=5: 5 > h[0] -> đẩy 2 ra, đưa 5 vào          heap=[3, 5] (h[0]=3)
i=4: x=6: 6 > h[0] -> đẩy 3 ra, đưa 6 vào          heap=[5, 6] (h[0]=5)
i=5: x=4: 4 <= h[0]=5 -> bỏ qua                    heap=[5, 6] (h[0]=5)
đáp án = h[0] = 5


5

In [17]:
def trace_quickselect(nums, k, choose=lambda arr: arr[len(arr) // 2]):
    arr = list(nums)
    round_no = 0
    print(f"nums={nums}, k={k}")
    while True:
        round_no += 1
        pivot = choose(arr)
        greater = [x for x in arr if x > pivot]
        equal = [x for x in arr if x == pivot]
        smaller = [x for x in arr if x < pivot]
        g, e = len(greater), len(equal)
        print(f"vòng {round_no}: arr={arr}, k={k}, pivot={pivot}")
        print(f"   greater={greater} (g={g}), equal={equal} (e={e}), smaller={smaller}")
        if k <= g:
            print(f"   k={k} <= g={g} -> đi vào nhóm greater")
            arr = greater
        elif k <= g + e:
            print(f"   {g} < k={k} <= {g + e} -> đáp án là pivot {pivot}")
            return pivot
        else:
            print(f"   k={k} > g+e={g + e} -> bỏ {g + e} số lớn, k := {k - g - e}, đi vào nhóm smaller")
            k -= g + e
            arr = smaller


trace_quickselect([3, 2, 3, 1, 2, 4, 5, 5, 6], 4)

nums=[3, 2, 3, 1, 2, 4, 5, 5, 6], k=4
vòng 1: arr=[3, 2, 3, 1, 2, 4, 5, 5, 6], k=4, pivot=2
   greater=[3, 3, 4, 5, 5, 6] (g=6), equal=[2, 2] (e=2), smaller=[1]
   k=4 <= g=6 -> đi vào nhóm greater
vòng 2: arr=[3, 3, 4, 5, 5, 6], k=4, pivot=5
   greater=[6] (g=1), equal=[5, 5] (e=2), smaller=[3, 3, 4]
   k=4 > g+e=3 -> bỏ 3 số lớn, k := 1, đi vào nhóm smaller
vòng 3: arr=[3, 3, 4], k=1, pivot=3
   greater=[4] (g=1), equal=[3, 3] (e=2), smaller=[]
   k=1 <= g=1 -> đi vào nhóm greater
vòng 4: arr=[4], k=1, pivot=4
   greater=[] (g=0), equal=[4] (e=1), smaller=[]
   0 < k=1 <= 1 -> đáp án là pivot 4


4

Bảng tóm tắt quickselect (ví dụ 2, `k = 4`, pivot ở giữa mảng hiện tại):

| vòng | mảng | `k` | pivot | `g` | `e` | quyết định |
|---|---|---|---|---|---|---|
| 1 | `[3,2,3,1,2,4,5,5,6]` | 4 | `2` | 6 | 2 | `k <= g` → vào `greater` |
| 2 | `[3,3,4,5,5,6]` | 4 | `5` | 1 | 2 | `k > g + e = 3` → bỏ 3 số lớn, `k := 1`, vào `smaller` |
| 3 | `[3,3,4]` | 1 | `3` | 1 | 2 | `k <= g` → vào `greater` |
| 4 | `[4]` | 1 | `4` | 0 | 1 | `0 < k <= 1` → đáp án `4` |

Bốn vòng, kích thước mảng giảm `9 → 6 → 3 → 1`, và các phần tử `1`, `2` không bao giờ phải sắp xếp. Chú ý vòng 2: khi đi vào nhóm `smaller` thì **phải trừ `g + e` khỏi `k`**.

## 9. Code nộp LeetCode

Bản nộp dùng **min-heap cỡ `k`** (Cách 2): đúng `class Solution`, đúng `findKthLargest(self, nums: List[int], k: int) -> int`, không làm thay đổi `nums`, và có đảm bảo **worst case** `O(n log k)`.

Bản quickselect tại chỗ (Cách 4) cũng được chấp nhận trên LeetCode (trung bình `O(n)`); ta để dạng class ở đây để bạn so sánh.

In [18]:
class Solution:
    def findKthLargest(self, nums: List[int], k: int) -> int:
        h = []                                  # min-heap giữ k số lớn nhất đã thấy
        for x in nums:
            if len(h) < k:
                heapq.heappush(h, x)
            elif x > h[0]:
                heapq.heapreplace(h, x)
        return h[0]                             # nhỏ nhất trong k số lớn nhất = lớn thứ k


class SolutionQuickselect:
    def findKthLargest(self, nums: List[int], k: int) -> int:
        lo, hi, target = 0, len(nums) - 1, k - 1
        while lo <= hi:
            p = nums[random.randint(lo, hi)]
            lt, i, gt = lo, lo, hi
            while i <= gt:
                if nums[i] > p:
                    nums[lt], nums[i] = nums[i], nums[lt]
                    lt += 1
                    i += 1
                elif nums[i] < p:
                    nums[i], nums[gt] = nums[gt], nums[i]
                    gt -= 1
                else:
                    i += 1
            if target < lt:
                hi = lt - 1
            elif target > gt:
                lo = gt + 1
            else:
                return p


sol = Solution()
for nums, k, expected in examples:
    assert sol.findKthLargest(nums, k) == expected
    assert SolutionQuickselect().findKthLargest(list(nums), k) == expected
print("OK: 2 ví dụ của đề (cả hai bản nộp)")

OK: 2 ví dụ của đề (cả hai bản nộp)


## 10. Vì sao thuật toán đúng

- **Sắp xếp:** sau khi sắp giảm dần, phần tử ở chỉ số `k-1` có đúng `k - 1` phần tử đứng trước (lớn hơn hoặc bằng) — đúng định nghĩa "lớn thứ `k`" kể cả khi có trùng.
- **Min-heap cỡ `k`:** *invariant:* sau khi xử lý `m` phần tử đầu, heap chứa **đúng `min(m, k)` phần tử lớn nhất** của chúng. Khi `x` đến và heap đã đủ `k`: nếu `x <= h[0]` thì `x` không thể nằm trong `k` số lớn nhất (đã có `k` số `>= h[0] >= x`; trong trường hợp bằng nhau việc giữ hay bỏ `x` cho cùng giá trị `h[0]`); nếu `x > h[0]` thì `h[0]` bị loại vì có `k` số lớn hơn nó. Sau `n` phần tử, `h[0]` là **nhỏ nhất của `k` số lớn nhất** = lớn thứ `k`.
- **Quickselect:** sau khi chia, mọi phần tử `greater` đứng trước (lớn hơn) mọi `equal`, mọi `equal` đứng trước `smaller`. Vị trí `k` (đếm từ 1) rơi vào `greater` nếu `k <= g`, vào `equal` nếu `g < k <= g + e` (đáp án là `pivot` vì cả khối `equal` có cùng giá trị), và vào `smaller` ở vị trí `k - g - e` nếu còn lại. Mỗi vòng giữ đúng bài toán con tương đương nên cuối cùng dừng (kích thước giảm vì `equal` luôn có ít nhất pivot).
- **Đếm:** duyệt giá trị từ lớn xuống nhỏ và trừ số lần xuất hiện chính là "đi qua các hạng theo thứ tự giảm dần"; khi `k` về `<= 0` thì ta vừa đi qua hạng thứ `k`.

## 11. Phân tích Time / Space Complexity

Gọi `n = len(nums)`, `R = max - min + 1` (tối đa `20001`).

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Sắp xếp | `O(n log n)` | `O(n)` | đơn giản, ổn định |
| 2. Min-heap cỡ `k` | `O(n log k)` | `O(k)` | worst case chắc chắn |
| 3. Heapify + `k-1` pop | `O(n + k log n)` | `O(n)` | tốt khi `k` nhỏ |
| 4. Quickselect | trung bình `O(n)`, worst `O(n^2)` | `O(1)` tại chỗ (hoặc `O(n)` danh sách) | worst case rất hiếm với pivot ngẫu nhiên |
| 5. Đếm | `O(n + R)` | `O(R)` | cần miền giá trị nhỏ |

Với `n = 10^5`: `n log n ≈ 1.7·10^6` phép, `R = 2·10^4`, tất cả đều rất nhẹ. Cân nhắc thực tế: khi giá trị trải rộng thì Cách 5 không dùng được; khi cần worst case đảm bảo thì Cách 2; khi muốn trung bình tuyến tính thì Cách 4.

## 12. Bộ test

`oracle` là brute force `O(n^2)` **độc lập, không sắp xếp**: phần tử lớn thứ `k` là **giá trị lớn nhất `x`** sao cho có **ít nhất `k`** phần tử `>= x`. Ta đối chiếu mọi cách với nó.

In [19]:
def oracle(nums, k):
    """Brute force độc lập: giá trị lớn nhất x trong nums mà số phần tử >= x là ít nhất k."""
    best = None
    for x in nums:
        count_ge = 0
        for y in nums:
            if y >= x:
                count_ge += 1
        if count_ge >= k and (best is None or x > best):
            best = x
    return best


impls = {
    "sort": kth_sort,
    "min_heap": kth_min_heap,
    "max_heap": kth_max_heap,
    "quickselect": kth_quickselect,
    "quickselect_inplace": kth_quickselect_inplace,
    "counting": kth_counting,
    "submit": sol.findKthLargest,
    "submit_quickselect": lambda nums, k: SolutionQuickselect().findKthLargest(list(nums), k),
}

for nums, k, expected in examples:
    assert oracle(nums, k) == expected
    for key, fn in impls.items():
        assert fn(list(nums), k) == expected, (key, nums, k, fn(list(nums), k), expected)
print("PASS ví dụ của đề (mọi cách và oracle)")

PASS ví dụ của đề (mọi cách và oracle)


### 12.1. Các trường hợp biên

In [20]:
edge_cases = [
    ([1], 1, 1, "một phần tử"),
    ([5, 5, 5, 5], 3, 5, "toàn bằng nhau"),
    ([2, 1], 1, 2, "k = 1 là max"),
    ([2, 1], 2, 1, "k = n là min"),
    ([-1, -1], 2, -1, "số âm, trùng"),
    ([-10000, 10000], 1, 10000, "biên của đề: giá trị lớn nhất"),
    ([-10000, 10000], 2, -10000, "biên của đề: giá trị nhỏ nhất"),
    ([7, 6, 5, 4, 3, 2, 1], 7, 1, "đã sắp giảm dần, k = n"),
    ([1, 2, 3, 4, 5, 6, 7], 1, 7, "đã sắp tăng dần, k = 1"),
    ([3, 3, 3, 1, 1], 4, 1, "lớn thứ 4 là 1 vì ba số 3 chiếm 3 hạng đầu"),
]
for nums, k, expected, label in edge_cases:
    assert oracle(nums, k) == expected, (label, oracle(nums, k), expected)
    for key, fn in impls.items():
        assert fn(list(nums), k) == expected, (key, label, nums, k, fn(list(nums), k), expected)
    print("PASS", label)

PASS một phần tử
PASS toàn bằng nhau
PASS k = 1 là max
PASS k = n là min
PASS số âm, trùng
PASS biên của đề: giá trị lớn nhất
PASS biên của đề: giá trị nhỏ nhất
PASS đã sắp giảm dần, k = n
PASS đã sắp tăng dần, k = 1
PASS lớn thứ 4 là 1 vì ba số 3 chiếm 3 hạng đầu


### 12.2. Tính chất và phản ví dụ

- Kết quả luôn **thuộc** mảng, và thỏa: có **ít nhất `k`** phần tử `>=` kết quả, và **nhiều nhất `k - 1`** phần tử `>` kết quả.
- Thêm một số rất lớn vào mảng và tăng `k` thêm 1 thì kết quả **không đổi**.
- **Phản ví dụ** cho cách sai dùng `set`: `[3,2,3,1,2,4,5,5,6]`, `k=4` cho đáp án `3` (sai) thay vì `4`.

In [21]:
random.seed(215_2)
for _ in range(400):
    n = random.randint(1, 30)
    nums = [random.randint(-6, 6) for _ in range(n)]
    k = random.randint(1, n)
    ans = sol.findKthLargest(nums, k)
    assert ans in nums
    assert sum(1 for y in nums if y >= ans) >= k
    assert sum(1 for y in nums if y > ans) <= k - 1
    assert sol.findKthLargest(nums + [10 ** 4], k + 1) == ans
print("PASS tính chất (thuộc mảng, đếm >= và >, thêm số rất lớn) trên 400 mảng")


def sai_set(nums, k):
    return sorted(set(nums), reverse=True)[k - 1]                  # SAI: bỏ trùng


nums, k = [3, 2, 3, 1, 2, 4, 5, 5, 6], 4
print("phản ví dụ: đúng", sol.findKthLargest(nums, k), "| dùng set:", sai_set(nums, k))
assert sai_set(nums, k) != sol.findKthLargest(nums, k)

PASS tính chất (thuộc mảng, đếm >= và >, thêm số rất lớn) trên 400 mảng
phản ví dụ: đúng 4 | dùng set: 3


### 12.3. Đối chiếu ngẫu nhiên (có seed)

In [22]:
random.seed(215)
for trial in range(2500):
    n = random.randint(1, 25)
    lo_v, hi_v = random.choice([(-3, 3), (-10, 10), (-10000, 10000), (5, 5)])
    nums = [random.randint(lo_v, hi_v) for _ in range(n)]
    k = random.randint(1, n)
    truth = oracle(nums, k)
    for key, fn in impls.items():
        assert fn(list(nums), k) == truth, (key, nums, k, fn(list(nums), k), truth)
print("PASS 2500 bộ ngẫu nhiên nhỏ (nhiều giá trị trùng, cả miền đầy đủ): mọi cách khớp oracle")

# Mảng lớn (n = 5000): so các cách với sắp xếp (oracle O(n^2) quá chậm ở cỡ này, nhưng sắp xếp đã được kiểm chứng ở trên)
for trial in range(20):
    n = 5000
    nums = [random.randint(-10000, 10000) for _ in range(n)]
    k = random.randint(1, n)
    ref = kth_sort(nums, k)
    for key, fn in impls.items():
        assert fn(list(nums), k) == ref, (key, k, fn(list(nums), k), ref)
print("PASS 20 mảng lớn (n = 5000, miền đầy đủ) khớp nhau")

PASS 2500 bộ ngẫu nhiên nhỏ (nhiều giá trị trùng, cả miền đầy đủ): mọi cách khớp oracle
PASS 20 mảng lớn (n = 5000, miền đầy đủ) khớp nhau


### 12.4. Đo thời gian thực tế (tham khảo)

Chỉ để cảm nhận: `n = 100 000`, miền giá trị đầy đủ. Con số tuyệt đối phụ thuộc máy, điều quan trọng là **thứ hạng tương đối**.

In [23]:
random.seed(100_000)
big = [random.randint(-10000, 10000) for _ in range(100_000)]
for kk in (10, 50_000):
    print(f"--- k = {kk}")
    for key in ("sort", "min_heap", "max_heap", "quickselect", "quickselect_inplace", "counting"):
        t0 = time.perf_counter()
        ans = impls[key](list(big), kk)
        dt = (time.perf_counter() - t0) * 1000
        print(f"{key:<20} ans={ans:>6}  {dt:8.1f} ms")

--- k = 10
sort                 ans=  9998      17.9 ms
min_heap             ans=  9998       4.0 ms
max_heap             ans=  9998       6.1 ms
quickselect          ans=  9998       9.1 ms
quickselect_inplace  ans=  9998      15.2 ms
counting             ans=  9998       5.4 ms
--- k = 50000
sort                 ans=   -32      16.5 ms
min_heap             ans=   -32      22.5 ms
max_heap             ans=   -32      29.5 ms
quickselect          ans=   -32      21.8 ms
quickselect_inplace  ans=   -32      25.3 ms
counting             ans=   -32       6.0 ms


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Dùng `set` để bỏ trùng
Xem `sai_set` ở phần test. **Vì sao sai:** đề nói rõ **không** phải phần tử phân biệt thứ `k`; các số trùng chiếm nhiều hạng. **Code đúng:** không loại trùng.

### Lỗi 2: Nhầm chỉ số `k` và `k - 1`
#### Code sai

In [24]:
def sai_chi_so(nums, k):
    return sorted(nums, reverse=True)[k]            # SAI: chỉ số k là phần tử thứ k+1


print("đúng:", Solution().findKthLargest([3, 2, 1, 5, 6, 4], 2), "| sai:", sai_chi_so([3, 2, 1, 5, 6, 4], 2))
try:
    sai_chi_so([1], 1)
except IndexError as e:
    print("k = n còn văng IndexError:", e)

đúng: 5 | sai: 4
k = n còn văng IndexError: list index out of range


**Vì sao sai:** chỉ số trong Python bắt đầu từ `0`; hạng `k` (đếm từ 1) nằm ở chỉ số `k - 1` (giảm dần) hoặc `-k` (tăng dần). **Code đúng:** `sorted(nums, reverse=True)[k - 1]`.

### Lỗi 3: Quên `heapq` là min-heap (giữ top `k` sai chiều)
#### Code sai

In [25]:
def sai_heap_chieu(nums, k):
    h = []
    for x in nums:
        heapq.heappush(h, -x)                        # SAI: lưu số đối rồi giữ cỡ k
        if len(h) > k:
            heapq.heappop(h)                         # pop phần tử nhỏ nhất của -x, tức số LỚN nhất của x!
    return -h[0]


print("đúng:", Solution().findKthLargest([3, 2, 1, 5, 6, 4], 2), "| sai:", sai_heap_chieu([3, 2, 1, 5, 6, 4], 2))

đúng: 5 | sai: 2


**Vì sao sai:** lưu `-x` rồi bỏ phần tử nhỏ nhất của `-x` là bỏ số **lớn nhất** của `x`; heap cuối cùng giữ `k` số **nhỏ** nhất. **Code đúng:** với heap cỡ `k` giữ top `k`, lưu **chính `x`** (min-heap); chỉ dùng `-x` khi `heapify` cả mảng rồi pop `k-1` lần (Cách 3).

### Lỗi 4: Quickselect chia 2 phần với nhiều số trùng (suy biến `O(n^2)` hoặc lặp vô hạn)
#### Code sai

In [26]:
def quickselect_2way_cmp(nums, k):
    """Chia hai phần (>= pivot và < pivot) sau khi tách pivot ra. Đếm số phép so sánh."""
    arr = list(nums)
    comparisons = 0
    while True:
        pivot = arr[0]
        rest = arr[1:]
        big = [x for x in rest if x >= pivot]              # gộp cả "bằng" vào nhóm lớn
        small = [x for x in rest if x < pivot]
        comparisons += len(rest)
        if k <= len(big):
            arr = big                                      # toàn số bằng nhau: chỉ loại được đúng 1 phần tử/vòng
        elif k == len(big) + 1:
            return pivot, comparisons
        else:
            k -= len(big) + 1
            arr = small


def quickselect_3way_rounds(nums, k):
    """Chia ba phần (>, ==, <): đếm số vòng."""
    arr, rounds = list(nums), 0
    while True:
        rounds += 1
        pivot = arr[0]
        greater = [x for x in arr if x > pivot]
        equal = [x for x in arr if x == pivot]
        if k <= len(greater):
            arr = greater
        elif k <= len(greater) + len(equal):
            return pivot, rounds
        else:
            k -= len(greater) + len(equal)
            arr = [x for x in arr if x < pivot]


all_equal = [7] * 2000
print("2 phần, toàn số bằng nhau, n=2000, k=1000: (đáp án, số phép so sánh) =", quickselect_2way_cmp(all_equal, 1000))
print("3 phần, cùng dữ liệu: (đáp án, số vòng) =", quickselect_3way_rounds(all_equal, 1000))

2 phần, toàn số bằng nhau, n=2000, k=1000: (đáp án, số phép so sánh) = (7, 1500499)
3 phần, cùng dữ liệu: (đáp án, số vòng) = (7, 1)


**Vì sao sai (chậm):** khi mọi số bằng nhau, chia hai phần chỉ loại được **một** phần tử mỗi vòng, tổng cỡ **hàng triệu** phép so sánh chỉ với `n = 2000` và tăng theo **bình phương** (với `n = 10^5` là hàng tỉ, quá thời gian). **Code đúng:** chia **ba phần** (`>`, `==`, `<`): khối `==` được xử lý một lần, đáp án tìm ra ngay.

### Lỗi 5: Chọn pivot cố định (phần tử đầu) trên mảng đã sắp xếp
Với mảng đã sắp xếp, pivot là phần tử đầu luôn là cực trị nên mỗi vòng chỉ loại được một phần tử: `O(n^2)`. **Cách sửa:** chọn pivot **ngẫu nhiên** (`random.choice`) hoặc "median of three".

In [27]:
def count_cmp_first_pivot(nums, k):
    arr, comps = list(nums), 0
    while True:
        pivot = arr[0]                                    # pivot cố định: phần tử đầu
        greater = [x for x in arr if x > pivot]
        equal = [x for x in arr if x == pivot]
        comps += len(arr)
        g, e = len(greater), len(equal)
        if k <= g:
            arr = greater
        elif k <= g + e:
            return pivot, comps
        else:
            k -= g + e
            arr = [x for x in arr if x < pivot]


def count_cmp_random_pivot(nums, k, rng):
    arr, comps = list(nums), 0
    while True:
        pivot = rng.choice(arr)                           # pivot ngẫu nhiên
        greater = [x for x in arr if x > pivot]
        equal = [x for x in arr if x == pivot]
        comps += len(arr)
        g, e = len(greater), len(equal)
        if k <= g:
            arr = greater
        elif k <= g + e:
            return pivot, comps
        else:
            k -= g + e
            arr = [x for x in arr if x < pivot]


asc = list(range(2000))
print("pivot đầu, mảng tăng dần, k=1000:", count_cmp_first_pivot(asc, 1000))
print("pivot ngẫu nhiên, cùng dữ liệu :", count_cmp_random_pivot(asc, 1000, random.Random(5)))

pivot đầu, mảng tăng dần, k=1000: (1000, 1501500)
pivot ngẫu nhiên, cùng dữ liệu : (1000, 6463)


**Vì sao khác nhau:** pivot cố định gặp mảng "xấu" thì thu hẹp rất chậm; pivot ngẫu nhiên khiến không có đầu vào nào xấu một cách ổn định (kỳ vọng `O(n)`).

## 14. Mẹo làm nhanh & nhận dạng pattern

- Thấy "**kth largest/smallest**", "top K", "K phần tử gần nhất/tần suất cao nhất" → nghĩ **heap cỡ `k`** (`O(n log k)`); với max/min lớn dùng heap ngược chiều: **top `k` lớn → min-heap cỡ `k`**, top `k` nhỏ → max-heap cỡ `k` (lưu số đối).
- Đề "**không sắp xếp**", "tuyến tính" → **Quickselect** (trung bình `O(n)`), nhớ pivot ngẫu nhiên và chia 3 phần.
- Giá trị **nhỏ, bị chặn** (như `|x| <= 10^4`) → **đếm / bucket**; đây là lợi thế của đọc kỹ constraints.
- `heapq.nlargest(k, nums)[-1]` là một dòng đúng nhưng ẩn chi tiết; nên hiểu cách heap làm việc trước khi dùng.
- Giữ ý: **hạng `k` tính từ 1** nhưng chỉ số tính từ 0.
- Kiểm tra bằng tay: `k = 1` (max), `k = n` (min), mảng có số trùng, số âm.

## 15. Biến thể của bài

1. **Kth Smallest** (nhỏ thứ `k`): đảo chiều so sánh, hoặc `kth_largest(nums, n - k + 1)`.
2. **Kth Largest Element in a Stream** (LeetCode 703): heap cỡ `k` giữ nguyên, thêm số liên tục.
3. **Top K Frequent Elements** (LeetCode 347): đếm tần suất rồi heap/bucket.
4. **K Closest Points to Origin** (LeetCode 973): heap với khóa là khoảng cách.
5. **Kth Smallest Element in a Sorted Matrix** (378) / **Find K Pairs with Smallest Sums** (373): heap trên nhiều dãy đã sắp.
6. **Median of Two Sorted Arrays** (4): quickselect/tìm kiếm nhị phân trên hạng.

In [28]:
def kth_smallest(nums, k):
    return kth_quickselect(nums, len(nums) - k + 1)


class KthLargest:
    """Biến thể LeetCode 703: luồng số, heap cỡ k."""
    def __init__(self, k, nums):
        self.k = k
        self.h = []
        for x in nums:
            self.add(x)

    def add(self, val):
        if len(self.h) < self.k:
            heapq.heappush(self.h, val)
        elif val > self.h[0]:
            heapq.heapreplace(self.h, val)
        return self.h[0]


print(kth_smallest([3, 2, 1, 5, 6, 4], 2))                 # 2
stream = KthLargest(3, [4, 5, 8, 2])
print([stream.add(x) for x in (3, 5, 10, 9, 4)])           # [4, 5, 5, 8, 8]

2
[4, 5, 5, 8, 8]


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Sắp xếp | sắp giảm dần, lấy `[k-1]` | `O(n log n)` | `O(n)` | ngắn gọn, `n` vừa phải |
| 2. Min-heap cỡ `k` | chỉ giữ `k` số lớn nhất | `O(n log k)` | `O(k)` | **bản nộp an toàn**, dữ liệu luồng |
| 3. Heapify + pop | max-heap bằng số đối, bỏ `k-1` số | `O(n + k log n)` | `O(n)` | `k` rất nhỏ |
| 4. Quickselect | chia theo pivot, đi vào một phía | TB `O(n)`, worst `O(n^2)` | `O(1)`/`O(n)` | đề cấm sắp xếp, muốn tuyến tính |
| 5. Đếm | đếm tần suất giá trị rồi duyệt từ lớn xuống | `O(n + R)` | `O(R)` | miền giá trị nhỏ (như đề này) |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: (a) `sorted([4, 1, 3, 3], reverse=True)[1]`; (b) `[x for x in [5, 2, 7, 2] if x > 2]`; (c) với `h = [3, 1, 2]` rồi `heapq.heapify(h)`, `h[0]` là bao nhiêu; (d) `Solution().findKthLargest([1, 1, 1], 3)`.

**Bài 2 (tính bằng tay).** Chạy tay min-heap cỡ `k = 3` trên `[5, 1, 9, 3, 7, 2]`: ghi heap sau mỗi phần tử và đáp án.

**Bài 3 (điền TODO).** Hoàn thiện `kth_todo(nums, k)` bằng min-heap cỡ `k`.

In [29]:
def kth_todo(nums, k):
    h = []
    # TODO 1: duyệt từng x trong nums
    # TODO 2: nếu heap chưa đủ k phần tử -> heapq.heappush(h, x)
    # TODO 3: ngược lại nếu x > h[0] -> heapq.heapreplace(h, x)
    # TODO 4: trả h[0]
    pass


# Tự làm xong rồi mới mở test:
# for nums, k, expected in examples:
#     assert kth_todo(nums, k) == expected

**Bài 4 (tự viết lại).** Viết `kth_quick2(nums, k)` bằng Quickselect với `random.choice` làm pivot và chia 3 phần (`greater`, `equal`, `smaller`), **không nhìn lại** Cách 4.

**Bài 5 (biến thể / mini challenge).** Viết `top_k_largest(nums, k)` trả về danh sách `k` phần tử lớn nhất **theo thứ tự giảm dần** (tính cả trùng) bằng min-heap cỡ `k`; ví dụ `top_k_largest([3,2,3,1,2,4,5,5,6], 4) == [6, 5, 5, 4]`. Sau đó tự trả lời: dùng cách nào nếu bài hỏi **tổng** của `k` số lớn nhất?

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: nhớ chỉ số bắt đầu từ `0`; `heapify` đưa số nhỏ nhất lên `h[0]`. Bài 2: heap giữ `k` số lớn nhất đã thấy, `h[0]` là nhỏ nhất trong chúng. Bài 5: sau khi xử lý xong, sắp xếp heap giảm dần.
- **Mức 2 (vừa):** Bài 1: `3`, `[5, 7]`, `1`, `1`. Bài 2: heap `[5]`, `[1,5]`, `[1,5,9]`; gặp `3 > 1`: thay -> `[3,5,9]`; gặp `7 > 3`: thay -> `[5,7,9]`; gặp `2 <= 5`: bỏ. Bài 3: `heappush` khi `len(h) < k`, `heapreplace` khi `x > h[0]`. Bài 4: xem Cách 4.
- **Mức 3 (sâu):** Bài 2: đáp án `h[0] = 5` (top 3 là `9, 7, 5`). Bài 5: `sorted(h, reverse=True)` sau vòng lặp. Tổng của `k` số lớn nhất: dùng cùng heap, đáp án là `sum(h)`; hoặc quickselect rồi cộng phần `>= kth`, cẩn thận phần tử trùng đúng bằng `kth`.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [30]:
# Bài 1
assert sorted([4, 1, 3, 3], reverse=True)[1] == 3
assert [x for x in [5, 2, 7, 2] if x > 2] == [5, 7]
h = [3, 1, 2]
heapq.heapify(h)
assert h[0] == 1
assert Solution().findKthLargest([1, 1, 1], 3) == 1

# Bài 2: top 3 của [5, 1, 9, 3, 7, 2] là 9, 7, 5 -> đáp án 5
assert Solution().findKthLargest([5, 1, 9, 3, 7, 2], 3) == 5 == oracle([5, 1, 9, 3, 7, 2], 3)


# Bài 3
def kth_todo_ans(nums, k):
    h = []
    for x in nums:
        if len(h) < k:
            heapq.heappush(h, x)
        elif x > h[0]:
            heapq.heapreplace(h, x)
    return h[0]


for nums, k, expected in examples:
    assert kth_todo_ans(nums, k) == expected


# Bài 4
def kth_quick2(nums, k):
    arr = list(nums)
    while True:
        pivot = random.choice(arr)
        greater = [x for x in arr if x > pivot]
        equal = [x for x in arr if x == pivot]
        if k <= len(greater):
            arr = greater
        elif k <= len(greater) + len(equal):
            return pivot
        else:
            k -= len(greater) + len(equal)
            arr = [x for x in arr if x < pivot]


random.seed(1)
for nums, k, expected in examples:
    assert kth_quick2(nums, k) == expected


# Bài 5
def top_k_largest(nums, k):
    h = []
    for x in nums:
        if len(h) < k:
            heapq.heappush(h, x)
        elif x > h[0]:
            heapq.heapreplace(h, x)
    return sorted(h, reverse=True)


assert top_k_largest([3, 2, 3, 1, 2, 4, 5, 5, 6], 4) == [6, 5, 5, 4]
assert sum(top_k_largest([3, 2, 3, 1, 2, 4, 5, 5, 6], 4)) == 20
print("PASS đáp án bài tập")

PASS đáp án bài tập


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (min-heap cỡ `k`)

```python
import heapq

class Solution:
    def findKthLargest(self, nums: List[int], k: int) -> int:
        h = []
        for x in nums:
            if len(h) < k:
                heapq.heappush(h, x)
            elif x > h[0]:
                heapq.heapreplace(h, x)
        return h[0]
```

**Checklist**

- [ ] Hiểu "lớn thứ `k`" **tính cả phần tử trùng** (không dùng `set`)?
- [ ] Chỉ số: hạng `k` ↔ chỉ số `k - 1` (giảm dần) hoặc `-k` (tăng dần)?
- [ ] `heapq` là **min-heap**: top-`k` lớn dùng min-heap cỡ `k`, đáp án `h[0]`?
- [ ] Quickselect: pivot **ngẫu nhiên**, chia **3 phần**, cập nhật `k` đúng khi đi vào nhóm nhỏ hơn?
- [ ] Đã đọc constraints (`n <= 10^5`, `|nums[i]| <= 10^4`) để cân nhắc cách đếm?
- [ ] Đã thử `k = 1`, `k = n`, mảng toàn bằng nhau, số âm?
- [ ] Đã đối chiếu với brute force `O(n^2)` độc lập trên mảng ngẫu nhiên?
- [ ] Skeleton đúng `class Solution` / `def findKthLargest(self, nums: List[int], k: int) -> int`?